# Exercise 3: A Vision Transformer for Landslide Mapping

Exercise 2 built a U-Net from scratch, a convolutional network trained entirely from randomly initialised weights. This exercise introduces a different family of architecture, the Vision Transformer, and a different training strategy, fine tuning a network that already carries pre trained weights.

The model used here is SegFormer, a segmentation architecture built around a Mix Transformer (MiT) backbone. We use the smallest variant, MiT b0, with weights obtained from pre training on ImageNet, and fine tune it for landslide mapping on the Landslide4Sense dataset.

## What you will learn

1. The conceptual difference between a convolutional network and a Vision Transformer
2. How SegFormer combines a hierarchical transformer backbone with a lightweight decoder
3. How to load a pre trained SegFormer checkpoint through the Hugging Face `transformers` library and adapt it to a new task
4. How to fine tune the network and evaluate it with Intersection over Union

## From local convolutions to global attention

A convolutional layer only looks at a small neighbourhood of pixels at a time, for example a three by three window. A network built from convolutions therefore needs many stacked layers before information from one side of an image can influence a prediction on the other side.

A transformer works differently. It splits the image into patches and lets every patch attend to every other patch through a mechanism called self attention, starting from the very first layer. Each patch effectively asks which other patches are relevant to understanding it, and combines information from across the whole image accordingly. This gives transformers access to global context immediately, rather than only after many layers.

## The Mix Transformer backbone

SegFormer does not use a single flat transformer. Instead, its backbone, called Mix Transformer or MiT, is organised into four stages that progressively reduce the spatial resolution while increasing the number of channels, in a similar spirit to a convolutional encoder.

| Stage | Spatial size (relative to input) | Typical channels | What it tends to capture |
|-------|-----------------------------------|-------------------|----------------------------|
| 1     | one quarter                        | 32                | fine texture and edges     |
| 2     | one eighth                          | 64                | local shapes                |
| 3     | one sixteenth                       | 160               | larger patterns             |
| 4     | one thirty second                   | 256               | broad scene layout          |

Each stage uses an efficient form of self attention that reduces the spatial size of the keys and values before computing attention, which keeps memory use manageable even though every patch can still attend globally within its stage.

## A lightweight decoder

Because every stage of the MiT backbone already has access to global context, SegFormer does not need a complex decoder with many skip connections, unlike U-Net. Its decoder, called the All MLP decode head, simply projects the features from every stage to the same number of channels, upsamples them to a common resolution, concatenates them, and applies a small convolutional classifier. The simplicity of the decoder is possible precisely because the backbone features are already rich.

## Dataset

Landslide4Sense provides 128 by 128 pixel patches assembled from Sentinel 2 optical bands together with slope and elevation derived from radar based terrain data. Each patch has 14 bands in total: Sentinel 2 bands B1 to B12, a slope band, and a digital elevation model band. Every pixel in the accompanying mask is labelled 0 for no landslide or 1 for landslide.

The pre trained SegFormer checkpoint we use expects a three channel input, since it was originally trained on ordinary colour photographs. To keep the fine tuning step straightforward and make full use of the pre trained weights, this exercise builds the model input from the three visible bands, Red, Green and Blue, which correspond to Sentinel 2 bands B4, B3 and B2. A note at the end of the exercise explains how the model could be extended to use all 14 bands.

In [ ]:
!pip install torch torchvision transformers huggingface_hub h5py matplotlib numpy -q

In [ ]:
import glob
import random

import numpy as np
import h5py
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import SegformerForSemanticSegmentation

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## Downloading the dataset

The dataset is stored on Hugging Face as a set of HDF5 files, organised into `images` and `annotations` folders, each split further into `train`, `validation` and `test`. The `snapshot_download` function retrieves the whole repository to a local folder.

In [ ]:
from huggingface_hub import snapshot_download

dataset_root = snapshot_download(repo_id="harshinde/LandSlide4Sense", repo_type="dataset", local_dir="landslide4sense")

train_image_files = sorted(glob.glob(f"{dataset_root}/images/train/*.h5"))
val_image_files = sorted(glob.glob(f"{dataset_root}/images/validation/*.h5"))

print(f"Training patches   : {len(train_image_files)}")
print(f"Validation patches : {len(val_image_files)}")

## Looking at a sample patch

Each image file stores its data under the key `img`, with shape 128 by 128 by 14. The matching mask file stores its data under the key `mask`. The cell below reads one patch, builds a true colour image from bands B4, B3 and B2, which are stored at array positions 3, 2 and 1 since band numbering starts at B1, and displays it next to the landslide mask.

In [ ]:
RGB_BAND_POSITIONS = [3, 2, 1]  # positions of B4 (Red), B3 (Green), B2 (Blue) in the 14 band stack


def read_patch(image_path):
    mask_path = image_path.replace("images", "annotations").replace("image_", "mask_")
    with h5py.File(image_path, "r") as f:
        image = f["img"][:]
    with h5py.File(mask_path, "r") as f:
        mask = f["mask"][:]
    return image, mask


def true_colour_for_display(image, upper_percentile=98):
    """Build a stretched Red, Green, Blue composite from the 14 band patch for plotting."""
    rgb = image[:, :, RGB_BAND_POSITIONS].astype(np.float32)
    upper_value = np.percentile(rgb, upper_percentile)
    return np.clip(rgb / upper_value, 0, 1)


sample_image, sample_mask = read_patch(train_image_files[0])

fig, axes = plt.subplots(1, 2, figsize=(9, 4.5))
axes[0].imshow(true_colour_for_display(sample_image))
axes[0].set_title("True colour (B4, B3, B2)")
axes[1].imshow(sample_mask, cmap="RdYlGn_r", vmin=0, vmax=1)
axes[1].set_title("Landslide mask")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

print(f"Image shape: {sample_image.shape}")
print(f"Fraction of landslide pixels in this patch: {sample_mask.mean():.3f}")

## Building a PyTorch dataset

The dataset class below reads the three visible bands from each patch and standardises them using the mean and standard deviation reported for the Landslide4Sense baseline model, so that the input values are centred around zero, similar to how the model's pre training data was prepared. The image is stored as a channel first tensor, since this is the layout PyTorch and the Hugging Face vision models expect.

In [ ]:
# Per band mean and standard deviation for bands B1 to B12, slope and elevation,
# taken from the published Landslide4Sense baseline.
BAND_MEAN = [-0.4914, -0.3074, -0.1277, -0.0625, 0.0439, 0.0803, 0.0644, 0.0802,
             0.3000, 0.4082, 0.0823, 0.0516, 0.3338, 0.7819]
BAND_STD = [0.9325, 0.8775, 0.8860, 0.8869, 0.8857, 0.8418, 0.8354, 0.8491,
            0.9061, 1.6072, 0.8848, 0.9232, 0.9018, 1.2913]


def random_flip_and_rotate(image, mask):
    """Apply the same random flip and 90 degree rotation to an image and its mask."""
    if random.random() < 0.5:
        image = torch.flip(image, dims=[2])
        mask = torch.flip(mask, dims=[1])
    if random.random() < 0.5:
        image = torch.flip(image, dims=[1])
        mask = torch.flip(mask, dims=[0])
    number_of_rotations = random.randint(0, 3)
    if number_of_rotations > 0:
        image = torch.rot90(image, number_of_rotations, dims=[1, 2])
        mask = torch.rot90(mask, number_of_rotations, dims=[0, 1])
    return image, mask


class LandslideDataset(Dataset):
    def __init__(self, image_files, augment=False):
        self.image_files = image_files
        self.augment = augment

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, index):
        image, mask = read_patch(self.image_files[index])

        rgb = image[:, :, RGB_BAND_POSITIONS].astype(np.float32)
        for channel, band_position in enumerate(RGB_BAND_POSITIONS):
            rgb[:, :, channel] = (rgb[:, :, channel] - BAND_MEAN[band_position]) / BAND_STD[band_position]

        rgb = np.transpose(rgb, (2, 0, 1))  # (3, 128, 128)

        image_tensor = torch.from_numpy(rgb)
        mask_tensor = torch.from_numpy(mask.astype(np.int64))

        if self.augment:
            image_tensor, mask_tensor = random_flip_and_rotate(image_tensor, mask_tensor)

        return image_tensor, mask_tensor


train_dataset = LandslideDataset(train_image_files, augment=True)
val_dataset = LandslideDataset(val_image_files, augment=False)

BATCH_SIZE = 16
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

sample_batch_images, sample_batch_masks = next(iter(train_loader))
print(f"Image batch shape: {sample_batch_images.shape}")
print(f"Mask batch shape : {sample_batch_masks.shape}")

## Loading the pre trained SegFormer

The `transformers` library downloads the MiT b0 backbone weights automatically the first time this cell runs. Because we ask for two output classes with `num_labels=2`, the library keeps the pre trained backbone but creates a new, randomly initialised decode head sized for our task. A warning about newly initialised weights is expected and simply confirms that the head was not part of the checkpoint being loaded.

Patches in this dataset are 128 by 128 pixels, which is already a multiple of 32, the total downsampling factor of the four backbone stages. Unlike some other exercises, no resizing of the input is required.

In [ ]:
model = SegformerForSemanticSegmentation.from_pretrained(
    "nvidia/mit-b0",
    num_labels=2,
    id2label={0: "no_landslide", 1: "landslide"},
    label2id={"no_landslide": 0, "landslide": 1},
).to(device)

backbone_parameters = sum(p.numel() for p in model.segformer.parameters())
head_parameters = sum(p.numel() for p in model.decode_head.parameters())
print(f"Backbone parameters (pre trained): {backbone_parameters:,}")
print(f"Decode head parameters (randomly initialised): {head_parameters:,}")

## Loss function and optimiser

SegFormer produces logits at a quarter of the input resolution, since the decode head operates on the backbone's first stage resolution rather than the full image. These logits need to be upsampled back to the mask resolution before the loss can be computed. The training and validation loop below follows the same pattern used in Exercise 2, so that the two exercises can be compared directly.

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=6e-5)


def run_one_epoch(model, loader, optimizer, criterion, device, training):
    model.train() if training else model.eval()

    total_loss = 0.0
    correct_pixels = 0
    total_pixels = 0

    torch.set_grad_enabled(training)
    for images, masks in loader:
        images = images.to(device)
        masks = masks.to(device)

        if training:
            optimizer.zero_grad()

        logits = model(pixel_values=images).logits
        upsampled_logits = nn.functional.interpolate(
            logits, size=masks.shape[-2:], mode="bilinear", align_corners=False
        )
        loss = criterion(upsampled_logits, masks)

        if training:
            loss.backward()
            optimizer.step()

        total_loss += loss.item() * images.size(0)

        predictions = torch.argmax(upsampled_logits, dim=1)
        correct_pixels += (predictions == masks).sum().item()
        total_pixels += masks.numel()

    average_loss = total_loss / len(loader.dataset)
    pixel_accuracy = correct_pixels / total_pixels
    return average_loss, pixel_accuracy

In [ ]:
NUM_EPOCHS = 10

history = {"train_loss": [], "val_loss": [], "train_accuracy": [], "val_accuracy": []}

for epoch in range(NUM_EPOCHS):
    train_loss, train_accuracy = run_one_epoch(model, train_loader, optimizer, criterion, device, training=True)
    val_loss, val_accuracy = run_one_epoch(model, val_loader, optimizer, criterion, device, training=False)

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["train_accuracy"].append(train_accuracy)
    history["val_accuracy"].append(val_accuracy)

    print(f"Epoch {epoch + 1:2d}/{NUM_EPOCHS}  "
          f"train loss {train_loss:.4f}  val loss {val_loss:.4f}  "
          f"train accuracy {train_accuracy:.4f}  val accuracy {val_accuracy:.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

axes[0].plot(history["train_loss"], label="Training loss")
axes[0].plot(history["val_loss"], label="Validation loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Cross entropy loss")
axes[0].set_title("Loss over fine tuning")
axes[0].legend()

axes[1].plot(history["train_accuracy"], label="Training accuracy")
axes[1].plot(history["val_accuracy"], label="Validation accuracy")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Pixel accuracy")
axes[1].set_title("Accuracy over fine tuning")
axes[1].legend()

plt.tight_layout()
plt.show()

## Evaluating with Intersection over Union

Landslides typically cover a small fraction of each patch, so pixel accuracy on its own is not a reliable indicator of performance. The function below computes IoU separately for the landslide class and the no landslide class.

In [ ]:
def per_class_iou(model, loader, device, num_classes=2):
    model.eval()
    intersection = np.zeros(num_classes)
    union = np.zeros(num_classes)

    with torch.no_grad():
        for images, masks in loader:
            images = images.to(device)
            masks = masks.to(device)

            logits = model(pixel_values=images).logits
            upsampled_logits = nn.functional.interpolate(
                logits, size=masks.shape[-2:], mode="bilinear", align_corners=False
            )
            predictions = torch.argmax(upsampled_logits, dim=1)

            for class_id in range(num_classes):
                predicted_class = predictions == class_id
                true_class = masks == class_id
                intersection[class_id] += (predicted_class & true_class).sum().item()
                union[class_id] += (predicted_class | true_class).sum().item()

    return intersection / np.clip(union, 1, None)


class_names = ["No landslide", "Landslide"]
iou_scores = per_class_iou(model, val_loader, device)
for name, score in zip(class_names, iou_scores):
    print(f"{name:12s} IoU: {score:.4f}")
print(f"Mean IoU: {iou_scores.mean():.4f}")

## Visualising predictions

The cell below displays the true colour composite, the ground truth mask and the model prediction for a few validation patches, which makes it possible to see whether errors tend to happen at landslide boundaries, on small isolated landslides, or in a more widespread way across the scene.

In [ ]:
def plot_predictions(model, image_files, device, num_samples=4, seed=3):
    model.eval()
    rng = np.random.RandomState(seed)
    indices = rng.choice(len(image_files), num_samples, replace=False)

    dataset_for_plotting = LandslideDataset(image_files, augment=False)

    fig, axes = plt.subplots(num_samples, 3, figsize=(10, 3.4 * num_samples))
    column_titles = ["True colour composite", "Ground truth", "Prediction"]
    for column, title in enumerate(column_titles):
        axes[0, column].set_title(title)

    with torch.no_grad():
        for row, index in enumerate(indices):
            raw_image, raw_mask = read_patch(image_files[index])
            image_tensor, mask_tensor = dataset_for_plotting[index]

            logits = model(pixel_values=image_tensor.unsqueeze(0).to(device)).logits
            upsampled_logits = nn.functional.interpolate(
                logits, size=raw_mask.shape, mode="bilinear", align_corners=False
            )
            prediction = torch.argmax(upsampled_logits, dim=1).squeeze(0).cpu().numpy()

            axes[row, 0].imshow(true_colour_for_display(raw_image))
            axes[row, 1].imshow(raw_mask, cmap="RdYlGn_r", vmin=0, vmax=1)
            axes[row, 2].imshow(prediction, cmap="RdYlGn_r", vmin=0, vmax=1)

            for ax in axes[row]:
                ax.axis("off")

    legend_handles = [
        mpatches.Patch(color="green", label="No landslide"),
        mpatches.Patch(color="red", label="Landslide"),
    ]
    fig.legend(handles=legend_handles, loc="lower center", ncol=2)
    plt.tight_layout(rect=[0, 0.03, 1, 1])
    plt.show()


plot_predictions(model, val_image_files, device)

## Extension: using all 14 bands

The main exercise uses only the three visible bands so that the pre trained SegFormer weights apply without modification. The additional Sentinel 2 bands, slope and elevation available in this dataset carry useful information for landslide detection, for example exposed soil and rock visible in the near infrared and shortwave infrared bands, and steep terrain visible in the slope band.

To use all 14 bands, the first layer of the backbone, which projects raw pixels into patch embeddings, needs to be replaced with a version that accepts 14 input channels instead of 3. The pre trained weights for the original three channels can be copied into the new layer, while the remaining eleven channels start from a small random initialisation. This lets the model keep most of its pre trained knowledge while learning to make use of the extra bands during fine tuning. The cell below shows how this replacement is done, without retraining the model further within this notebook.

In [ ]:
import copy

original_projection = model.segformer.encoder.patch_embeddings[0].proj
new_projection = nn.Conv2d(
    in_channels=14,
    out_channels=original_projection.out_channels,
    kernel_size=original_projection.kernel_size,
    stride=original_projection.stride,
    padding=original_projection.padding,
)

with torch.no_grad():
    new_projection.weight[:, RGB_BAND_POSITIONS, :, :] = original_projection.weight
    new_projection.bias[:] = original_projection.bias

extended_model = copy.deepcopy(model)
extended_model.segformer.encoder.patch_embeddings[0].proj = new_projection

print("New first layer now accepts", new_projection.in_channels, "input channels")
print("Weights for the three original bands were copied from the pre trained checkpoint")

## Summary

This exercise fine tuned a Vision Transformer instead of training a network from scratch. Loading the pre trained MiT b0 backbone gave the model a useful starting point learned from millions of natural images, so that only the small decode head, and a comparatively short fine tuning run, were needed to adapt it to landslide mapping.

Compared with the U-Net in Exercise 2, SegFormer reaches its result with a much smaller number of parameters trained from scratch, since most of the network is reused rather than learned anew. Exercise 5 returns to this comparison in more detail, using the learning curves and evaluation metrics introduced in both exercises.